*GPU Programming - Aljoscha Rheinwalt <aljoscha.rheinwalt@uni-potsdam.de>*

---

# U-Net Geology Mapping Example

This notebook is a self-contained example for binary geological mapping with Sentinel-2 raster data and a U-Net in PyTorch.

The workflow has four parts:

1. **Data**: read GeoTIFF rasters, create 128 x 128 training tiles, and save them in HDF5.
2. **Model**: define a small U-Net with skip connections.
3. **Training**: train with weighted BCE + Dice loss and a spatial validation split.
4. **Prediction**: slide the model over the full raster and plot probabilities with mapped-label contours.

## Table of Contents

- [Setup](#Setup)
  - Define file paths, imports, tile size, and run flags for expensive steps.
- [Data](#Data)
  - [Raster Helpers](#Raster-Helpers): open GeoTIFFs, inspect metadata, read windows, and compute hillshade.
  - [Preview the Rasters](#Preview-the-Rasters): view Sentinel-2 RGB data with the mapped target contour.
  - [Build Training Tiles](#Build-Training-Tiles): create 128 x 128 normalized tiles and save them to HDF5.
- [Model](#Model)
  - Define the U-Net encoder, decoder, skip connections, and output logits.
- [Training](#Training)
  - Load HDF5 tiles, make a spatial validation split, train with weighted BCE + Dice loss, and save the best checkpoint by class-1 IoU.
- [Prediction](#Prediction)
  - Stream full-raster tiles through the model, blend overlapping predictions, and save probability maps.
  - [Plot Prediction with Map Context](#Plot-Prediction-with-Map-Context): plot probabilities with DSM hillshade and cyan target contours.
  - [Interpreting the Result](#Interpreting-the-Result): read high-probability areas as expert-review candidates.


## Setup

Set the file names and training options here. The computationally expensive cells are guarded by flags, so the notebook can be opened and read without immediately rebuilding data or retraining the model.

In [ ]:
from pathlib import Path
import json
import math
import os
import h5py
import matplotlib.pyplot as pl
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm
from osgeo import gdal

In [ ]:
pl.rc("figure", dpi=150)
gdal.UseExceptions()

FEATURE_TIF = Path("S2_L2A_20250827_allbands_epsg32719_PunaECordillera_clip.tif")
TARGET_TIF = Path("e250K_ECordilleraClip_Puncoviscana_epsg32719.tif")
DSM_TIF = Path("COP_10m_PunaECordillera_int16_epsg32719.tif")
TRAIN_H5 = Path("train_data_puncoviscana_north.h5")
CHECKPOINT = Path("unet_geology.pt")
META_JSON = Path("unet_geology_meta.json")
PROBA_NPY = Path("prediction_proba.npy")

TILE_SIZE = 128
TILE_OFFSETS = ((0, 0), (64, 64))

RUN_DATA_BUILD = False
RUN_TRAINING = True
RUN_PREDICTION = True

## Data

The feature raster is a Sentinel-2 stack. The target raster is binary: pixels with value `1` are Puncoviscana and pixels with value `0` are background. For training we use only the northern half of the raster. That avoids a suspicious southern line in targets, while later prediction can still be run over the full feature raster.

### Raster Helpers

Open a raster, inspect metadata, read a full raster or a window, and make a hillshade from the DSM for plotting.

In [ ]:
def open_raster(path):
    ds = gdal.Open(str(path))
    if ds is None:
        raise FileNotFoundError(path)
    return ds


def raster_info(path):
    ds = open_raster(path)
    gt = ds.GetGeoTransform()
    info = {
        "path": str(path),
        "width": ds.RasterXSize,
        "height": ds.RasterYSize,
        "bands": ds.RasterCount,
        "dtype": gdal.GetDataTypeName(ds.GetRasterBand(1).DataType),
        "pixel_width": gt[1],
        "pixel_height": gt[5],
    }
    ds = None
    return info


def read_raster(path):
    ds = open_raster(path)
    arr = ds.ReadAsArray()
    ds = None
    if arr.ndim == 3 and arr.shape[0] == 1:
        arr = arr[0]
    return arr


def read_window(path, row, col, height, width=None):
    if width is None:
        width = height
    ds = open_raster(path)
    arr = ds.ReadAsArray(col, row, width, height)
    ds = None
    return arr


def progress(items, total=None, desc=None):
    if tqdm is None:
        return items
    return tqdm(items, total=total, desc=desc)


def hillshade_from_dsm(dsm, azimuth=315, altitude=45):
    dsm = dsm.astype(np.float32)
    gy, gx = np.gradient(dsm)
    slope = np.pi / 2 - np.arctan(np.hypot(gx, gy))
    aspect = np.arctan2(-gx, gy)
    az = np.deg2rad(azimuth)
    alt = np.deg2rad(altitude)
    shaded = np.sin(alt) * np.sin(slope) + np.cos(alt) * np.cos(slope) * np.cos(az - aspect)
    return np.clip((shaded + 1.0) / 2.0, 0.0, 1.0)

In [ ]:
for path in [FEATURE_TIF, TARGET_TIF, DSM_TIF]:
    print(raster_info(path))

### Preview the Rasters

Reads a small window and plots a simple RGB composite with the binary geology target as a cyan contour.

In [ ]:
def stretch_rgb(rgb, low=2, high=98):
    lo, hi = np.percentile(rgb, [low, high])
    return np.clip((rgb - lo) / max(hi - lo, 1e-6), 0.0, 1.0)


preview_size = 1024
feature_ds = open_raster(FEATURE_TIF)
preview_row = 1024
preview_col = 9216
feature_ds = None

x_preview = read_window(FEATURE_TIF, preview_row, preview_col, preview_size).astype(np.float32)
y_preview = read_window(TARGET_TIF, preview_row, preview_col, preview_size)

rgb = np.stack([x_preview[3], x_preview[2], x_preview[1]], axis=-1)
rgb = stretch_rgb(rgb)

fig, ax = pl.subplots(figsize=(12, 12))
ax.imshow(rgb)
ax.contour(y_preview > 0, levels=[0.5], colors="cyan", linewidths=2)
ax.set_title("Sentinel-2 preview with mapped Puncoviscana contour")
ax.axis("off")
pl.show()

### Build Training Tiles

The model sees small image tiles, not the full raster. Each feature tile has shape `(bands, 128, 128)` and each target tile has shape `(1, 128, 128)`.

The important choices are:

- **Northern half only**: training uses rows `0 : height // 2`.
- **Half-tile offset**: we use normal grid tiles and a second grid shifted by 64 pixels, which roughly doubles training examples.
- **Per-band min/max normalization**: each Sentinel-2 band is scaled to `[0, 1]` using min/max values measured over the selected training tiles.
- **HDF5 output**: training later reads only the HDF5 file, so GDAL is not needed on the training machine.

In [ ]:
def tile_origins(height, width, tile_size=TILE_SIZE, tile_offsets=TILE_OFFSETS):
    for row_offset, col_offset in tile_offsets:
        for row in range(row_offset, height - tile_size + 1, tile_size):
            for col in range(col_offset, width - tile_size + 1, tile_size):
                yield row, col


def count_tile_origins(height, width, tile_size=TILE_SIZE, tile_offsets=TILE_OFFSETS):
    total = 0
    for row_offset, col_offset in tile_offsets:
        n_rows = len(range(row_offset, height - tile_size + 1, tile_size))
        n_cols = len(range(col_offset, width - tile_size + 1, tile_size))
        total += n_rows * n_cols
    return total


def read_feature_tile(ds, row, col, tile_size=TILE_SIZE):
    x = ds.ReadAsArray(col, row, tile_size, tile_size)
    if x.ndim == 2:
        x = x[None, :, :]
    return x


def read_target_tile(ds, row, col, tile_size=TILE_SIZE):
    y = ds.ReadAsArray(col, row, tile_size, tile_size)
    if y.ndim == 3:
        y = y[0]
    return y > 0


def update_min_max(x, min_value, max_value):
    flat = x.reshape(x.shape[0], -1)
    return np.minimum(min_value, flat.min(axis=1)), np.maximum(max_value, flat.max(axis=1))


def normalize_feature_tile(x, min_value, max_value):
    x = x.astype(np.float32)
    min_value = np.asarray(min_value, dtype=np.float32)
    max_value = np.asarray(max_value, dtype=np.float32)
    scale = np.where(max_value - min_value < 1e-6, 1.0, max_value - min_value)
    x = (x - min_value[:, None, None]) / scale[:, None, None]
    return np.clip(x, 0.0, 1.0)

In [ ]:
def scan_tiles_for_normalization(feature_ds, height, width):
    n_bands = feature_ds.RasterCount
    feature_nodata = feature_ds.GetRasterBand(1).GetNoDataValue()
    min_value = np.full(n_bands, np.inf, dtype=np.float32)
    max_value = np.full(n_bands, -np.inf, dtype=np.float32)
    rows = []
    cols = []

    total = count_tile_origins(height, width)
    origins = tile_origins(height, width)
    for row, col in progress(origins, total=total, desc="scan tiles"):
        x = read_feature_tile(feature_ds, row, col).astype(np.float32)
        if feature_nodata is not None and not np.any(x != feature_nodata):
            continue
        min_value, max_value = update_min_max(x, min_value, max_value)
        rows.append(row)
        cols.append(col)

    return np.asarray(rows, dtype=np.int32), np.asarray(cols, dtype=np.int32), min_value, max_value


def build_training_h5(output_path=TRAIN_H5, max_tiles=None):
    feature_ds = open_raster(FEATURE_TIF)
    target_ds = open_raster(TARGET_TIF)

    height = min(feature_ds.RasterYSize, target_ds.RasterYSize) // 2
    width = min(feature_ds.RasterXSize, target_ds.RasterXSize)
    n_bands = feature_ds.RasterCount

    rows, cols, min_value, max_value = scan_tiles_for_normalization(feature_ds, height, width)
    if max_tiles is not None:
        rows = rows[:max_tiles]
        cols = cols[:max_tiles]

    with h5py.File(output_path, "w") as f:
        x_ds = f.create_dataset(
            "x",
            shape=(len(rows), n_bands, TILE_SIZE, TILE_SIZE),
            dtype=np.float16,
            compression="gzip",
            compression_opts=9,
            shuffle=True,
            chunks=(1, n_bands, TILE_SIZE, TILE_SIZE),
        )
        y_ds = f.create_dataset(
            "y",
            shape=(len(rows), 1, TILE_SIZE, TILE_SIZE),
            dtype=np.uint8,
            compression="gzip",
            compression_opts=9,
            shuffle=True,
            chunks=(1, 1, TILE_SIZE, TILE_SIZE),
        )

        positives = 0
        positions = zip(rows, cols)
        for i, (row, col) in enumerate(progress(positions, total=len(rows), desc="write tiles")):
            x = read_feature_tile(feature_ds, int(row), int(col))
            y = read_target_tile(target_ds, int(row), int(col))
            x_ds[i] = normalize_feature_tile(x, min_value, max_value).astype(np.float16)
            y_ds[i, 0] = y.astype(np.uint8)
            positives += int(y.sum())

        f.create_dataset("tile_row", data=rows)
        f.create_dataset("tile_col", data=cols)
        f.attrs["feature"] = str(FEATURE_TIF)
        f.attrs["target"] = str(TARGET_TIF)
        f.attrs["tile_size"] = TILE_SIZE
        f.attrs["tile_offsets"] = TILE_OFFSETS
        f.attrs["target_is_binary"] = True
        f.attrs["positive_classes"] = [1]
        f.attrs["normalization"] = "per_band_min_max_0_1_closed"
        f.attrs["normalization_min"] = min_value
        f.attrs["normalization_max"] = max_value
        f.attrs["source_rows"] = height
        f.attrs["source_cols"] = width
        f.attrs["source_region"] = "northern_half"

    feature_ds = None
    target_ds = None
    return len(rows), positives

In [ ]:
if RUN_DATA_BUILD:
    n_tiles, positives = build_training_h5(TRAIN_H5)
    pixels = n_tiles * TILE_SIZE * TILE_SIZE
    print(f"wrote: {TRAIN_H5}")
    print(f"tiles: {n_tiles}")
    print(f"positive pixels: {positives} / {pixels} ({positives / pixels:.4%})")
elif TRAIN_H5.exists():
    with h5py.File(TRAIN_H5, "r") as f:
        print("x", f["x"].shape, f["x"].dtype)
        print("y", f["y"].shape, f["y"].dtype)
        print("positive pixels", f["y"][:].mean())
        print("source region", f.attrs.get("source_region"))
        print("normalization", f.attrs.get("normalization"))
else:
    print("Set RUN_DATA_BUILD = True to create the HDF5 training file.")

## Model

A U-Net is an encoder-decoder segmentation model.

- The **encoder** repeatedly applies convolutions and pooling, so it learns increasingly abstract features at lower resolution.
- The **decoder** upsamples those features back to the original image size.
- **Skip connections** concatenate encoder features into the decoder. This helps the model combine coarse context with sharp spatial detail.

The model returns raw logits. We apply `torch.sigmoid` only when converting logits into probabilities.

In [ ]:
def conv_block(in_ch, out_ch):
    return nn.Sequential(
        nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
        nn.BatchNorm2d(out_ch),
        nn.ReLU(inplace=True),
        nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1, bias=False),
        nn.BatchNorm2d(out_ch),
        nn.ReLU(inplace=True),
    )


class UNet(nn.Module):
    def __init__(self, in_ch=12, out_ch=1, base=32):
        super().__init__()
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        self.enc1 = conv_block(in_ch, base)
        self.enc2 = conv_block(base, base * 2)
        self.enc3 = conv_block(base * 2, base * 4)
        self.enc4 = conv_block(base * 4, base * 8)

        self.bottleneck = conv_block(base * 8, base * 16)

        self.dec4 = conv_block(base * 16 + base * 8, base * 8)
        self.dec3 = conv_block(base * 8 + base * 4, base * 4)
        self.dec2 = conv_block(base * 4 + base * 2, base * 2)
        self.dec1 = conv_block(base * 2 + base, base)

        self.out = nn.Conv2d(base, out_ch, kernel_size=1)

    def forward(self, x):
        s1 = self.enc1(x)
        s2 = self.enc2(self.pool(s1))
        s3 = self.enc3(self.pool(s2))
        s4 = self.enc4(self.pool(s3))

        x = self.bottleneck(self.pool(s4))

        x = F.interpolate(x, size=s4.shape[-2:], mode="bilinear", align_corners=False)
        x = self.dec4(torch.cat([x, s4], dim=1))

        x = F.interpolate(x, size=s3.shape[-2:], mode="bilinear", align_corners=False)
        x = self.dec3(torch.cat([x, s3], dim=1))

        x = F.interpolate(x, size=s2.shape[-2:], mode="bilinear", align_corners=False)
        x = self.dec2(torch.cat([x, s2], dim=1))

        x = F.interpolate(x, size=s1.shape[-2:], mode="bilinear", align_corners=False)
        x = self.dec1(torch.cat([x, s1], dim=1))

        return self.out(x)

In [ ]:
model = UNet(in_ch=12, out_ch=1, base=32)
n_params = sum(p.numel() for p in model.parameters())
print(f"parameters: {n_params:,}")

x = torch.zeros(1, 12, 128, 128)
with torch.no_grad():
    logits = model(x)
    prob = torch.sigmoid(logits)

print("input", tuple(x.shape))
print("logits", tuple(logits.shape))
print("probability range", float(prob.min()), float(prob.max()))

## Training

The target class is much less common than background, so pixel accuracy alone is misleading. Training uses:

- **Weighted BCE with logits**: makes false negatives on the rare positive class more expensive.
- **Dice loss**: encourages overlap of predicted and true positive regions.
- **Spatial validation stripes**: validation tiles are contiguous row intervals, with a one-tile buffer removed from training to avoid overlap leakage.
- **Threshold sweep**: validation reports IoU for class 1 at several probability thresholds.

In [ ]:
def set_seed(seed=0):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def load_training_data(path=TRAIN_H5):
    with h5py.File(path, "r") as f:
        x = torch.from_numpy(f["x"][...]).float()
        y = torch.from_numpy(f["y"][...]).float()
        meta = dict(f.attrs)
        meta["tile_row"] = torch.from_numpy(f["tile_row"][...]).long()
        meta["tile_col"] = torch.from_numpy(f["tile_col"][...]).long()
    return x, y, meta


def dice_loss(logits, target, eps=1e-6):
    prob = torch.sigmoid(logits)
    dims = (1, 2, 3)
    intersection = (prob * target).sum(dim=dims)
    union = prob.sum(dim=dims) + target.sum(dim=dims)
    dice = (2.0 * intersection + eps) / (union + eps)
    return 1.0 - dice.mean()


def make_loss(pos_weight, dice_weight=0.5):
    bce = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    def loss_fn(logits, target):
        return bce(logits, target) + dice_weight * dice_loss(logits, target)

    return loss_fn

In [ ]:
def choose_validation_stripes(tile_row, tile_size, val_fraction, n_stripes, seed):
    min_row = int(tile_row.min().item())
    max_row = int(tile_row.max().item())
    total_height = max_row + tile_size - min_row
    stripe_height = max(tile_size, int(round(total_height * val_fraction / n_stripes)))
    possible_starts = torch.unique(tile_row).cpu().numpy().astype(int).tolist()

    rng = np.random.default_rng(seed)
    rng.shuffle(possible_starts)
    intervals = []
    for start in possible_starts:
        end = start + stripe_height
        if end > max_row + tile_size:
            continue
        overlaps = any(
            start < old_end + tile_size and end + tile_size > old_start
            for old_start, old_end in intervals
        )
        if overlaps:
            continue
        intervals.append((start, end))
        if len(intervals) == n_stripes:
            break

    if not intervals:
        raise RuntimeError("Could not create validation stripes.")
    return sorted(intervals)


def spatial_train_val_split(tile_row, tile_size, val_fraction=0.1, n_stripes=5, seed=0):
    intervals = choose_validation_stripes(tile_row, tile_size, val_fraction, n_stripes, seed)

    val_mask = torch.zeros(len(tile_row), dtype=torch.bool, device=tile_row.device)
    buffer_mask = torch.zeros(len(tile_row), dtype=torch.bool, device=tile_row.device)
    for start, end in intervals:
        val_mask |= (tile_row >= start) & (tile_row < end)
        buffer_mask |= (tile_row + tile_size > start) & (tile_row < end + tile_size)

    buffer_mask &= ~val_mask
    train_mask = ~(val_mask | buffer_mask)

    train_idx = torch.where(train_mask)[0]
    val_idx = torch.where(val_mask)[0]
    buffer_idx = torch.where(buffer_mask)[0]
    return train_idx, val_idx, buffer_idx, intervals

In [ ]:
def batch_indices(indices, batch_size, shuffle=False):
    if shuffle:
        indices = indices[torch.randperm(len(indices), device=indices.device)]
    for start in range(0, len(indices), batch_size):
        yield indices[start : start + batch_size]


def augment_batch(x, y):
    k = torch.randint(0, 4, ()).item()
    x = torch.rot90(x, k, dims=(2, 3))
    y = torch.rot90(y, k, dims=(2, 3))

    if torch.rand(()) < 0.5:
        x = torch.flip(x, dims=(3,))
        y = torch.flip(y, dims=(3,))

    if torch.rand(()) < 0.5:
        x = torch.flip(x, dims=(2,))
        y = torch.flip(y, dims=(2,))

    return x, y


def train_epoch(model, x, y, train_idx, batch_size, opt, loss_fn, augment=True):
    model.train()
    losses = []
    for idx in batch_indices(train_idx, batch_size, shuffle=True):
        x_batch = x[idx]
        y_batch = y[idx]
        if augment:
            x_batch, y_batch = augment_batch(x_batch, y_batch)

        opt.zero_grad(set_to_none=True)
        logits = model(x_batch)
        loss = loss_fn(logits, y_batch)
        loss.backward()
        opt.step()
        losses.append(loss.detach())

    return torch.stack(losses).mean().item()

In [ ]:
@torch.no_grad()
def evaluate(model, x, y, val_idx, batch_size, loss_fn, thresholds=(0.1, 0.3, 0.5, 0.7, 0.9)):
    model.eval()
    losses = []
    counts = {float(t): {"tp": 0.0, "fp": 0.0, "fn": 0.0, "tn": 0.0} for t in thresholds}

    for idx in batch_indices(val_idx, batch_size):
        logits = model(x[idx])
        losses.append(loss_fn(logits, y[idx]).detach())
        prob = torch.sigmoid(logits)
        target = y[idx] >= 0.5

        for threshold in counts:
            pred = prob >= threshold
            counts[threshold]["tp"] += (pred & target).sum().item()
            counts[threshold]["fp"] += (pred & ~target).sum().item()
            counts[threshold]["fn"] += (~pred & target).sum().item()
            counts[threshold]["tn"] += (~pred & ~target).sum().item()

    eps = 1e-9
    by_threshold = {}
    for threshold, c in counts.items():
        tp, fp, fn, tn = c["tp"], c["fp"], c["fn"], c["tn"]
        iou_0 = tn / max(tn + fp + fn, eps)
        iou_1 = tp / max(tp + fp + fn, eps)
        by_threshold[threshold] = {
            "accuracy": (tp + tn) / max(tp + fp + fn + tn, eps),
            "iou_0": iou_0,
            "iou_1": iou_1,
            "miou": 0.5 * (iou_0 + iou_1),
            "dice": (2.0 * tp) / max(2.0 * tp + fp + fn, eps),
        }

    best_threshold = max(by_threshold, key=lambda t: by_threshold[t]["iou_1"])
    return {
        "loss": torch.stack(losses).mean().item(),
        "threshold": best_threshold,
        "by_threshold": by_threshold,
        **by_threshold[best_threshold],
    }

In [ ]:
def save_checkpoint(path_pt, path_json, model, meta):
    torch.save({"model_state": model.state_dict(), "meta": meta}, path_pt)
    with open(path_json, "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2)


def train_model(
    data_path=TRAIN_H5,
    checkpoint_path=CHECKPOINT,
    meta_path=META_JSON,
    epochs=50,
    batch_size=16,
    lr=3e-4,
    weight_decay=1e-4,
    base=32,
    seed=0,
    val_fraction=0.1,
    val_stripes=5,
    val_thresholds=(0.1, 0.3, 0.5, 0.7, 0.9),
    dice_weight=0.5,
    augment=True,
):
    set_seed(seed)
    x, y, data_meta = load_training_data(data_path)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if device.type == "cuda":
        torch.backends.cudnn.benchmark = True
        torch.set_float32_matmul_precision("high")
    print("device:", device)

    x = x.to(device)
    y = y.to(device)
    tile_row = data_meta["tile_row"].to(device)

    train_idx, val_idx, buffer_idx, val_intervals = spatial_train_val_split(
        tile_row,
        tile_size=int(data_meta["tile_size"]),
        val_fraction=val_fraction,
        n_stripes=val_stripes,
        seed=seed,
    )

    pos = y[train_idx].sum().item()
    neg = y[train_idx].numel() - pos
    pos_weight_value = neg / max(pos, 1.0)

    model = UNet(in_ch=x.shape[1], out_ch=1, base=base).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    pos_weight = torch.tensor([pos_weight_value], dtype=torch.float32, device=device)
    loss_fn = make_loss(pos_weight=pos_weight, dice_weight=dice_weight)

    meta = {
        "model": "UNet",
        "in_ch": int(x.shape[1]),
        "out_ch": 1,
        "base": int(base),
        "tile_size": int(data_meta["tile_size"]),
        "target": str(data_meta["target"]),
        "feature": str(data_meta["feature"]),
        "positive_classes": np.asarray(data_meta["positive_classes"]).astype(int).tolist(),
        "target_is_binary": bool(data_meta.get("target_is_binary", True)),
        "normalization": str(data_meta["normalization"]),
        "normalization_min": np.asarray(data_meta["normalization_min"]).astype(float).tolist(),
        "normalization_max": np.asarray(data_meta["normalization_max"]).astype(float).tolist(),
        "training_data": str(data_path),
        "seed": int(seed),
        "split": "random_row_stripes_with_one_tile_buffer",
        "split_val_intervals": [(int(a), int(b)) for a, b in val_intervals],
        "train_tiles": int(len(train_idx)),
        "val_tiles": int(len(val_idx)),
        "buffer_tiles": int(len(buffer_idx)),
        "train_positive_fraction": float(y[train_idx].mean().item()),
        "val_positive_fraction": float(y[val_idx].mean().item()),
        "pos_weight": float(pos_weight_value),
        "dice_weight": float(dice_weight),
        "val_thresholds": list(val_thresholds),
        "augmentation": "random_rot90_hflip_vflip" if augment else "none",
        "checkpoint_metric": "val_iou_1",
    }

    print(f"tiles: train={len(train_idx)} val={len(val_idx)} buffer={len(buffer_idx)} total={len(x)}")
    print("validation row intervals:", val_intervals)
    print(f"positive pixels: train={meta['train_positive_fraction']:.4%} val={meta['val_positive_fraction']:.4%}")
    print(f"class balance: pos={pos:.0f} neg={neg:.0f} pos_weight={pos_weight_value:.3f}")

    best_iou_1 = -math.inf
    history = []
    for epoch in range(1, epochs + 1):
        train_loss = train_epoch(model, x, y, train_idx, batch_size, opt, loss_fn, augment=augment)
        metrics = evaluate(model, x, y, val_idx, batch_size, loss_fn, thresholds=val_thresholds)
        history.append({"epoch": epoch, "train_loss": train_loss, **metrics})

        threshold_text = " ".join(
            f"iou_1@{threshold:.2f}={metrics['by_threshold'][threshold]['iou_1']:.4f}"
            for threshold in val_thresholds
        )
        print(
            f"epoch {epoch:03d} | train_loss {train_loss:.5f} | val_loss {metrics['loss']:.5f} | "
            f"val_thr {metrics['threshold']:.2f} | val_dice {metrics['dice']:.4f} | "
            f"val_iou_0 {metrics['iou_0']:.4f} | val_iou_1 {metrics['iou_1']:.4f} | "
            f"val_miou {metrics['miou']:.4f} | val_acc {metrics['accuracy']:.4f} | {threshold_text}"
        )

        if metrics["iou_1"] > best_iou_1:
            best_iou_1 = metrics["iou_1"]
            save_checkpoint(checkpoint_path, meta_path, model, {**meta, "best_val": metrics})
            print(f"  saved {checkpoint_path} (best val_iou_1 {best_iou_1:.4f} at threshold {metrics['threshold']:.2f})")

    return model, history, meta

In [ ]:
if RUN_TRAINING:
    model, history, train_meta = train_model(TRAIN_H5, epochs=50)
elif META_JSON.exists():
    with open(META_JSON, "r", encoding="utf-8") as f:
        train_meta = json.load(f)
    print("training data", train_meta["training_data"])
    print("train tiles", train_meta["train_tiles"])
    print("val tiles", train_meta["val_tiles"])
    print("validation intervals", train_meta["split_val_intervals"])
    print("pos_weight", round(train_meta["pos_weight"], 3))
    print("best threshold", train_meta["best_val"]["threshold"])
    print("best val_iou_1", round(train_meta["best_val"]["iou_1"], 4))
else:
    print("Set RUN_TRAINING = True to train a model.")

## Prediction

Prediction uses the trained model on the full feature raster. Because the raster is large, we stream 128 x 128 tiles from GDAL instead of loading all bands into memory. Overlapping predictions are blended into one smooth window. This reduces visible tile seams compared with placing non-overlapping tile predictions side by side.

In [ ]:
def read_feature_tile_for_prediction(ds, row, col, tile_size=TILE_SIZE):
    h = ds.RasterYSize
    w = ds.RasterXSize
    read_h = min(tile_size, h - row)
    read_w = min(tile_size, w - col)
    x = ds.ReadAsArray(col, row, read_w, read_h)
    if x.ndim == 2:
        x = x[None, :, :]
    if read_h == tile_size and read_w == tile_size:
        return x
    return np.pad(x, ((0, 0), (0, tile_size - read_h), (0, tile_size - read_w)), mode="edge")


def tile_starts(length, tile_size=TILE_SIZE, stride=64):
    starts = list(range(0, max(length - tile_size + 1, 1), stride))
    last = length - tile_size
    if starts[-1] != last:
        starts.append(last)
    return starts


def tile_weight(tile_size=TILE_SIZE):
    window = np.hanning(tile_size)
    weight = np.outer(window, window).astype(np.float32)
    weight /= weight.max()
    return np.maximum(weight, 0.05)


@torch.no_grad()
def predict_probability_raster(model, feature_ds, min_value, max_value, tile_size, stride, batch_size, device):
    h = feature_ds.RasterYSize
    w = feature_ds.RasterXSize
    rows = tile_starts(h, tile_size, stride)
    cols = tile_starts(w, tile_size, stride)
    positions = [(row, col) for row in rows for col in cols]

    prob_sum = np.zeros((h, w), dtype=np.float32)
    weight_sum = np.zeros((h, w), dtype=np.float32)
    weight = tile_weight(tile_size)

    model.eval()
    batch_tiles = []
    batch_positions = []

    def flush_batch():
        if not batch_tiles:
            return
        batch = torch.from_numpy(np.stack(batch_tiles)).float().to(device)
        logits = model(batch)
        probs = torch.sigmoid(logits).squeeze(1).cpu().numpy()
        for prob, (r, c) in zip(probs, batch_positions):
            r_end = min(r + tile_size, h)
            c_end = min(c + tile_size, w)
            dh = r_end - r
            dw = c_end - c
            prob_sum[r:r_end, c:c_end] += prob[:dh, :dw] * weight[:dh, :dw]
            weight_sum[r:r_end, c:c_end] += weight[:dh, :dw]
        batch_tiles.clear()
        batch_positions.clear()

    for row, col in progress(positions, total=len(positions), desc="predict tiles"):
        x = read_feature_tile_for_prediction(feature_ds, row, col, tile_size)
        x = normalize_feature_tile(x, min_value, max_value)
        batch_tiles.append(x)
        batch_positions.append((row, col))
        if len(batch_tiles) == batch_size:
            flush_batch()

    flush_batch()
    return prob_sum / np.maximum(weight_sum, 1e-6)

In [ ]:
def load_model_for_prediction(checkpoint_path=CHECKPOINT):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    checkpoint = torch.load(checkpoint_path, map_location=device)
    meta = checkpoint["meta"]

    model = UNet(
        in_ch=int(meta["in_ch"]),
        out_ch=int(meta.get("out_ch", 1)),
        base=int(meta.get("base", 32)),
    ).to(device)
    model.load_state_dict(checkpoint["model_state"])
    return model, meta, device


def run_full_prediction(checkpoint_path=CHECKPOINT, output_npy=PROBA_NPY, stride=None, batch_size=16):
    model, meta, device = load_model_for_prediction(checkpoint_path)
    feature_ds = open_raster(meta["feature"])
    stride = stride or int(meta["tile_size"]) // 2

    prob = predict_probability_raster(
        model=model,
        feature_ds=feature_ds,
        min_value=meta["normalization_min"],
        max_value=meta["normalization_max"],
        tile_size=int(meta["tile_size"]),
        stride=stride,
        batch_size=batch_size,
        device=device,
    )
    feature_ds = None
    np.save(output_npy, prob)
    print(f"wrote: {output_npy}")
    print(f"probability range: {prob.min():.4f} to {prob.max():.4f}")
    return prob

In [ ]:
if RUN_PREDICTION:
    prob = run_full_prediction(CHECKPOINT, PROBA_NPY, stride=64, batch_size=16)
elif PROBA_NPY.exists():
    prob = np.load(PROBA_NPY)
    print("loaded", PROBA_NPY, prob.shape, prob.dtype, float(prob.min()), float(prob.max()))
else:
    print("Set RUN_PREDICTION = True to create prediction_proba.npy.")

### Plot Prediction with Map Context

The plot shows model probability in `magma_r`, true mapped Puncoviscana as a cyan contour, and a hillshade computed from the DSM.

In [ ]:
def choose_plot_step(shape, max_size=5000, pixel_step=None):
    if pixel_step is not None:
        return max(1, pixel_step)
    return max(1, int(np.ceil(max(shape) / max_size)))


def plot_probability(prob, target_path=TARGET_TIF, dsm_path=DSM_TIF, max_size=5000, pixel_step=None):
    label = read_raster(target_path)
    label = label[: prob.shape[0], : prob.shape[1]] > 0

    dsm = read_raster(dsm_path)
    dsm = dsm[: prob.shape[0], : prob.shape[1]]

    step = choose_plot_step(prob.shape, max_size=max_size, pixel_step=pixel_step)
    prob_small = prob[::step, ::step]
    label_small = label[::step, ::step]
    dsm_small = dsm[::step, ::step]
    hs = hillshade_from_dsm(dsm_small)

    fg, ax = pl.subplots(figsize=(14, 14))
    prob_image = ax.imshow(prob_small, cmap="magma_r", vmin=0.0, vmax=1.0)
    ax.imshow(hs, cmap="gray", alpha=0.35)
    ax.contour(label_small, levels=[0.5], colors="cyan", linewidths=0.8)
    fg.colorbar(prob_image, ax=ax, label="Predicted probability", aspect=100)
    ax.set_title(f"Prediction probability with mapped target contour; display step = {step}")
    ax.axis("off")
    pl.show()

In [ ]:
plot_probability(prob)

### Interpreting the Result

The cyan line is the mapped Puncoviscana label. The color image is the model's predicted probability. High-probability areas outside the cyan outline are not automatically mistakes; they are useful candidates for expert review, especially in the southern part that was excluded from training.